### Cell 1: Importing Essential Libraries 

In [4]:
import os
import glob
from PIL import Image
import shutil
import numpy as np
import cv2
import pandas as pd
import warnings
import shutil
warnings.simplefilter("ignore")
import tensorflow.keras.layers as Layers
import tensorflow.keras.activations as Actications
import tensorflow.keras.models as Models
import tensorflow.keras.optimizers as Optimizer
import tensorflow.keras.metrics as Metrics
import tensorflow.keras.utils as Utils
import matplotlib.pyplot as plt
from sklearn.utils import shuffle
from sklearn.metrics import confusion_matrix as CM
from random import randint
import matplotlib.gridspec as gridspec
from tensorflow.keras.models import model_from_json
import glob,os.path
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Input, Conv2D,Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from IPython.display import clear_output
import random, re, math
import tensorflow.keras.backend as K
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import zipfile
from tensorflow.keras.models import model_from_json
from datetime import datetime
import mysql.connector 
import threading
import time
from tmd_bmd_old import *

ModuleNotFoundError: No module named 'cv2'

### Cell 2: Loading Pretrained Models and Configuration

- Change working directory.
- Load head slice classification model.
- Load masking model.
- Configure image size, epochs, seed, and batch size.


In [2]:
os.chdir('/home/castrone/PRIMUS/data/83_BIOINFORMATICS/Julia/Python') 
#NN_model_classify_head_slice1



json_file_head_slice_1176 = open('NN_model_classify_head_1176_retrained.json', 'r')
loaded_model_json_head_slice_1176 = json_file_head_slice_1176.read()
json_file_head_slice_1176.close()

loaded_model_head_slice_1176 = model_from_json(loaded_model_json_head_slice_1176)
loaded_model_head_slice_1176.load_weights("NN_model_classify_head_1176_retrained.h5")
print("Model for head slice 1176 classification was loadded!")

json_file_masking_1176 = open('NN_model_masking_1176.json', 'r')
loaded_model_json_masking_1176 = json_file_masking_1176.read()
json_file_masking_1176.close()
loaded_model_masking_1176 = model_from_json(loaded_model_json_masking_1176)
# load weights into new model
loaded_model_masking_1176.load_weights("NN_model_masking_1176.h5")
print("Model for masking 1176 was loadded!")

json_file_holder_masking_1176 = open('NN_model_masking_right_holder_1176.json', 'r')
loaded_model_json_holder_masking_1176 = json_file_holder_masking_1176.read()
json_file_holder_masking_1176.close()
loaded_model_holder_masking_1176 = model_from_json(loaded_model_json_holder_masking_1176)
# load weights into new model
loaded_model_holder_masking_1176.load_weights("NN_model_masking_right_holder_1176.h5")
print("Model for holder masking was loadded!")

AUTO = tf.data.experimental.AUTOTUNE

# Configuration
IMAGE_SIZE = [256, 256]
EPOCHS = 40
SEED = 777
BATCH_SIZE = 16 

FileNotFoundError: [WinError 3] The system cannot find the path specified: '/home/castrone/PRIMUS/data/83_BIOINFORMATICS/Julia/Python'

### CELL 3: SCRIPT FOR ANALYSING FOLDER FOR DEAD MICE 

In [6]:
def analyse_folder(parent):
    """
    Analyze a directory and generate CSV files with file metadata.

    This function recursively processes files within the specified 'parent' directory and its subdirectories. It extracts file metadata, such as ID, scanner information, file path, image width, and image height, and creates CSV files to store this metadata.

    Args:
    - parent (str): The path to the parent directory to be analyzed.

    Returns:
    - None

    CSV Files:
    - Two types of CSV files are generated:
      1. CSV files for 'died' mice, identified by filenames ending with '+'. Each 'died' mouse has a separate CSV file containing its metadata.
      2. CSV files for regular BMP image files, excluding specific file types and folder names.

    Notes:
    - The function skips specific file types and directories based on naming conventions.
    - The generated CSV files are stored in a subdirectory named 'csv_files_check_new_old' within the 'cohort_folder' path.

    Example:
            >>> analyse_folder('/path/to/parent_directory')

    This function will process the specified directory and create CSV files based on the metadata of the files it contains.

    """
    df = pd.DataFrame(columns = ['ID', 'SCANNER', 'PATH', 'IMAGE_WIDTH', 'IMAGE_HEIGHT'])
    ID = []
    scanner = []
    path = []
    image_width = []
    image_height = []
    split = []
    id_mouse = ''
    
    df_died = pd.DataFrame(columns = ['ID', 'SCANNER', 'PATH', 'IMAGE_WIDTH', 'IMAGE_HEIGHT'])
    ID_died = []
    scanner_died = []
    path_died = []
    image_width_died = []
    image_height_died = []
    id_mouse_died = ''
    
    isExist = os.path.exists(cohort_folder+'/'+'csv_files_check_new_old')
    if not isExist:
        os.mkdir(cohort_folder+'/'+'csv_files_check_new_old')
        
    for file_name in sorted(os.listdir(parent)):
        if file_name == 'Ignore':
            continue
        
        if file_name.endswith('body') or file_name.endswith('fantom1') or file_name.endswith('fantom2') or file_name.endswith('RecDXA') or file_name.endswith('mask') or file_name.endswith('Morphology') or file_name.endswith('head') or file_name.endswith('mask_edited') or file_name.endswith('predictions'):
            continue
            
        if file_name.endswith('+'):
            id_mouse_died= file_name.split('+')[0]
            ID_died.append(id_mouse_died)  
            scanner_died.append("DIED")
            path = "".join((parent,'/', file_name))
            path_died.append(path)
            image_width_died.append(None)
            image_height_died.append(None)
            
        if len(ID_died)>0:
            df_died['ID']=ID_died[:1]
            df_died['SCANNER']=scanner_died[:1]
            df_died['PATH']=path_died[:1]
            df_died['IMAGE_WIDTH']=image_width_died[:1]
            df_died['IMAGE_HEIGHT']=image_height_died[:1]
            df_died.to_csv(cohort_folder+'/'+'csv_files_check_new_old'+'/'+id_mouse_died+'+'+'.csv',index=False)   
            
        if file_name.endswith('.bmp') and 'VOI' not in parent and '_without_fantoms' not in parent:
            im = cv2.imread(parent + '/' + file_name)
            h, w, c = im.shape
            split= file_name.split('_')
            id_mouse = file_name.split('rec')[0]
            id_mouse = id_mouse.split('__')[0]
            
            image_width.append(w)
            image_height.append(h)
            #print(id_mouse)
            

            ID.append(id_mouse)  
            scanner.append(1176)
            path.append(parent)
 
        else:
            current_path = "".join((parent,'/', file_name))
            if os.path.isdir(current_path):
                print(current_path)
                analyse_folder(current_path)
            else:
                continue 
             
    if len(ID)>0:
        df['ID']=ID[:1]
        df['SCANNER']=scanner[:1]
        df['PATH']=path[:1]
        df['IMAGE_WIDTH']=image_width[:1]
        df['IMAGE_HEIGHT']=image_height[:1]
        print(df)
   
    if len(df)>0:
        df.to_csv(cohort_folder+'/'+'csv_files_check_new_old'+'/'+id_mouse+'.csv',index=False)      

### Cell 4: Setting Cohort Folder and Cohort Number

In [ ]:
#cohort_folder='/home/potip/PRIMUS/data/83_IMPC_DATA/Bioimaging/SCREENad.Morpho&BodyComp/Screen20220214c227_Apobec1 & Cactin & Cops8 & Pank2 & Wdr63 & wt'
cohort_folder = '/home/castrone/PRIMUS/data/83_IMPC_DATA/Bioimaging/SCREENad.Morpho&BodyComp/Screen20210426c188_Fam53a & Krtap9-5 & Ror1 & Slc38a8KO & Sox14 & wt' # change cogort folder here 
cohort_number = cohort_folder.split('c')[3] # Change here in case your username does not have a 'c'
cohort_number = cohort_number.split('_')[0]
cohort_number = 'c'+ cohort_number
print('Cohort folder:',cohort_folder)
print('Cohort number:',cohort_number)

### Cell 5: Analyzing Data and Consolidating CSV Files

In [ ]:
analyse_folder(cohort_folder)
print("done")
#find all csv files
all_files = glob.glob(os.path.join(cohort_folder+'/'+'csv_files_check_new_old', "*.csv"))
##create one dataframe with all csv files
df_from_each_file = (pd.read_csv(f, sep=',') for f in all_files)
df_cohort   = pd.concat(df_from_each_file, ignore_index=True)
#create one csv file
df_cohort.to_csv(cohort_folder+'/'+"Check_1278_1176_died.csv",index=False)
#shutil.rmtree(cohort_folder+'/'+'csv_files_check_new_old')

### Cell 6: Reading and Sorting Consolidated Data

In [ ]:
df_cohort = pd.read_csv(cohort_folder+'/'+"Check_1278_1176_died.csv")
df_cohort.sort_values(by=['ID'], inplace=True)
df_cohort

### Cell 7: RUN THIS EVERY COHORT - If needed, change '1278' or '1176' to 1278 or 1176

In [ ]:
df_cohort["IMAGE_WIDTH"] = pd.to_numeric(df_cohort["IMAGE_WIDTH"])
df_cohort["IMAGE_HEIGHT"] = pd.to_numeric(df_cohort["IMAGE_HEIGHT"])
df_cohort_1176 = df_cohort.loc[df_cohort['SCANNER']=='1176'] # If df_cohort_1176 is empty, change the '' situation
print('Data Frame for Scanner 1176')
display(df_cohort_1176)



### Cell 8: RUN THIS EVERY COHORT - Check if you need to remove items from the dataframe and if they are complete

In [ ]:
#df_cohort_1176 = df_cohort_1176.iloc[5], df_cohort_1176.iloc[11] # if you want to get specific elements, use iloc, like this line

redo = ['wt_M_82Z-06626', 'Fam53a_M_E64-F9379']


#df_cohort_1278 = df_cohort_1278.loc[df_cohort_1278['ID'] == '*.bmp'] # You can use this syntax to get all elements that don't have (!=) or have (==) one specific ID, IMAGE_WIDHT, IMAGE_HEIGHT


#df_cohort_1278 = df_cohort_1278[-2:-1] # This line will return the cohort elements until the 6th element. 
# You can also use something like df_cohort_1278 = df_cohort_1278[:-1] to return all elements, except the last.
# The syntax is this: df_cohort_1278['starting element (included)':'last element (excluded)']

#df_cohort_1278 = pd.concat([df_cohort_1278[:11], df_cohort_1278[14:]]) # run this to join different slices of the dataframe
# You can join more than two slices, just put them inside the [ ]. Like so: pd.concat([df_cohort_1278[:5], df_cohort_1278[7:15], df_cohort_1278[18:22], df_cohort_1278[25:-1]])

#df_cohort_1176 = df_cohort_1176[-1:] # Just like df_cohort_1278

#df_cohort_1176 = pd.concat([df_cohort_1176[1:2], df_cohort_1176[3:]]) # Just like df_cohort_1278

display(df_cohort_1176) # Use this line to show the df_cohort_1176 below

# Cell 10: 1176 - RUN THIS FOR EVERY COHORT THAT HAS 1176 ANIMALS

In [ ]:
for path in df_cohort_1176['PATH']:
    
    print('############## Part 1: function to predict the head slice - Running ###########')
    init = time.time()
    predict_head_slice_1176(path,loaded_model_head_slice_1176)
    
    end = time.time()
    print("############################This step took " + str(end-init) + " seconds###########################")
    print('############## Part 1 - Finished ###########')
    
    
    print('############## Part 2: function for masking body without head - Running ###########')
    init = time.time()

    selected_row = df_cohort_1176.loc[df_cohort_1176['PATH'] == path]
    masking_predict_1176(path, int(selected_row['IMAGE_WIDTH']), int(selected_row['IMAGE_HEIGHT']),loaded_model_masking_1176)

    end = time.time()
    print("############################This step took " + str(end-init) + " seconds###########################")
    print('############## Part 2 - Finished ###########')
    
    
    print('############## Part 3: function to mask holder - Running ###########')
    init = time.time()

    selected_row = df_cohort_1176.loc[df_cohort_1176['PATH'] == path]
    holder_masking_predict(path, int(selected_row['IMAGE_WIDTH']), int(selected_row['IMAGE_HEIGHT']),loaded_model_holder_masking_1176) 

    end = time.time()
    print("############################This step took " + str(end-init) + " seconds###########################")
    print('############## Part 3 - Finished ###########')
    
    
    print('############## Part 4: function for BMD and TMD calculation - Running ###########')
    init = time.time()

    isBMD_TMD = os.path.exists(cohort_folder+'/'+'BMD_TMD')
    if not isBMD_TMD:
        os.mkdir(cohort_folder+'/'+'BMD_TMD')
    TMD_BMD_1176(path,cohort_folder)

    end = time.time()
    print("############################This step took " + str(end-init) + " seconds###########################")
    print('############## Part 4 - Running ###########')

### Cell 11: Run this to create the DXA file. Check if all the live animals have values and if the values are ok

In [ ]:
all_files = glob.glob(os.path.join(cohort_folder+'/'+'BMD_TMD', "*.csv")) ################# CHANGE HERE
df_from_each_file = (pd.read_csv(f, sep=',',header=0) for f in all_files)
df_merged = pd.concat(df_from_each_file, ignore_index=True)

df_merged.drop(df_merged.filter(regex="Unname"),axis=1, inplace=True)
df_merged = df_merged.sort_values(by='CCP_mouseID')
df_merged = df_merged.reset_index(drop=True)
df_merged.rename(columns = {'CCP_mouseID':'Identificator','CCP_date':'CCP_DXA_001', 'CCP_metadata':'','CCP_DXA_793_001':'','CCP_DXA_004_001':''}, inplace = True)
df_merged.loc[-1] = ['CCP_mouseID','CCP_date', 'CCP_metadata','CCP_DXA_793_001','CCP_DXA_004_001']
df_merged.index = df_merged.index + 1
df_merged.loc[-1] = ['MouseID','Date', 'MetaDatalink','TMD','BMD']
df_merged.index = df_merged.index + 1
df_merged = df_merged.sort_index()
#print(df_merged)
df_merged.to_excel(cohort_folder+'/'+'DXA '+cohort_number+'.xlsx',index=False,index_label=False)


# connect to the DB
conn = mysql.connector.connect(host='london02.img.local', port=3306, database='vestlontest', user='vestlontest_bi', password='vJPeF5T6a83j$w8N')
cursor = conn.cursor()
sql= "SELECT * FROM MOUSE WHERE COHORT_ID ='%s'"%(cohort_number)
cursor.execute(sql)
database=pd.DataFrame(cursor.fetchall(),columns=['DOB','ID','SACRIFICE', 'LAB_ID', 'COHORT_ID','CAGE_ID'])
cursor.close()

# Read the excel file
sheet1_df = pd.read_excel(cohort_folder+'/'+'DXA '+cohort_number+'.xlsx')

# transform DF
excel_transf = sheet1_df[['Identificator', 'CCP_DXA_001']][2:]
excel_transf = excel_transf.reset_index(drop=True)
excel_transf=excel_transf.rename(columns={"Identificator":"ID", 'CCP_DXA_001':'Folder_date'})

#transform database table
database = database.sort_values(by='ID')
database = database[['ID','SACRIFICE']]

#find ids that are not in excel table
died=database['ID'][~database['ID'].isin(excel_transf['ID'])].dropna()
died = died.tolist()
for idd in died:
    new_row = {'Identificator':idd, 'CCP_DXA_001':'2021-12-13', 'Unnamed: 2':'UCT 0245', 'Unnamed: 3':'IMPC_PSC_001', 'Unnamed: 4':'IMPC_PSC_001'}
    sheet1_df = sheet1_df.append(new_row, ignore_index=True)
#    
#for i in df_cohort_1176['ID_without'].values:
    #sheet1_df.loc[sheet1_df['Identificator'] == i, 'Unnamed: 2'] = 'UCT 0241'     
    
#print(sheet1_df)    
# add ids with code    
sheet1_df = sheet1_df[2:].sort_values(by='Identificator')
sheet1_df = sheet1_df.reset_index()
sheet1_df.pop('index')
sheet1_df.loc[-1] = ['CCP_mouseID','CCP_date', 'CCP_metadata','CCP_DXA_793_001','CCP_DXA_004_001']
sheet1_df.index = sheet1_df.index + 1
sheet1_df.loc[-1] = ['MouseID','Date', 'MetaDatalink','TMD','BMD']
sheet1_df.index = sheet1_df.index + 1
sheet1_df = sheet1_df.sort_index()
sheet1_df=sheet1_df.rename(columns={"Unnamed: 2":" ", 'Unnamed: 3':'', 'Unnamed: 4':''})

Python_version = !python -V

# Create DataFrame
if len(df_cohort_1176['ID'])>0:
    sheet2_df = pd.DataFrame({'MetaData':['CCP_metadata', 'UCT 0241', 'UCT 0245'], 
                              'Date equipment last calibrated':['CCP_DXA_017_001', excel_transf['Folder_date'][0], excel_transf['Folder_date'][0]],
                              'Operator: Rec':['CCP_DXA_999_000', 'DXA_SD_009', 'DXA_SD_009'],#DXA_SD_009
                              'Operator: ROI':['CCP_DXA_016_001', 'DXA_sfw_009','DXA_sfw_009'],
                              'Operator: BMD':['CCP_DXA_996_000', 'DXA_sfw_009','DXA_sfw_009'],
                              'Equipment ID':['CCP_DXA_011_001', '19G14030', '14G08101'], #14G08101
                              'Equipment manufacturer':['CCP_DXA_012_001', 'Bruker','Bruker'],#Bruker
                              'Equipment model':['CCP_DXA_013_001', 'SkyScan 1176', 'SkyScan 1278'],
                              'Mouse Status':['CCP_DXA_014_001', 'Anesthetized', 'Anesthetized'],
                              'Anesthesia':['CCP_DXA_015_001', 'Zoletil+Xylazine', 'Zoletil+Xylazine'],
                              'DXA_Analytical software producer':['CCP_DXA_993_001', 'Python Software Foundation', 'Python Software Foundation'],
                              'DXA_Analytical software':['CCP_DXA_992_001', 'Python', 'Python'],
                              'DXA_Analytical software version':['CCP_DXA_991_001', Python_version[0], Python_version[0]]})
    print(sheet1_df)
else:
    sheet2_df = pd.DataFrame({'MetaData':['CCP_metadata', 'UCT 0245'], 
                              'Date equipment last calibrated':['CCP_DXA_017_001', excel_transf['Folder_date'][0]],
                              'Operator: Rec':['CCP_DXA_999_000', 'DXA_SD_009'],#DXA_SD_009
                              'Operator: ROI':['CCP_DXA_016_001', 'DXA_sfw_009'],
                              'Operator: BMD':['CCP_DXA_996_000', 'DXA_sfw_009'],
                              'Equipment ID':['CCP_DXA_011_001', '19G14030'], #14G08101
                              'Equipment manufacturer':['CCP_DXA_012_001', 'Bruker'],#Bruker
                              'Equipment model':['CCP_DXA_013_001', 'SkyScan 1278'],
                              'Mouse Status':['CCP_DXA_014_001', 'Anesthetized'],
                              'Anesthesia':['CCP_DXA_015_001', 'Zoletil+Xylazine'],
                              'DXA_Analytical software producer':['CCP_DXA_993_001', 'Python Software Foundation'],
                              'DXA_Analytical software':['CCP_DXA_992_001', 'Python'],
                              'DXA_Analytical software version':['CCP_DXA_991_001', Python_version[0]]})
    print(sheet1_df)



    
with pd.ExcelWriter('/home/castrone/PRIMUS/data/83_IMPC_DATA/Bioimaging/STATISTICS/_FINISHED_ Tabs_'+'/'+'DXA '+cohort_number+'.xlsx') as writer:
    sheet1_df.to_excel(writer,sheet_name="Sheet1",index=False,index_label=False)
    sheet2_df.to_excel(writer,sheet_name="Sheet2",index=False,index_label=False)

<span style="color:blue"><font size="5">Cell 12: Part 1 for 1278 - *function to detect the fantoms*</span>.

<span style="color:blue"><font size="5">Cell 13: Part 2 for 1278 - *function to predict the head slice*</span>.

<span style="color:blue"><font size="5">Cell 14: Part 3 for 1278 - *function for masking body without head*</span>.

<span style="color:blue"><font size="5">Cell 15: Part 4 for 1278 - *function for masking fantoms to get ROI*</span>.

<span style="color:blue"><font size="5">Cell 16: Part 5 - *function for BMD and TMD calculation*</span>.

<span style="color:green"><font size="5">Cell 17: Part 1 for 1176 - *function to predict the head slice*</span>.

<span style="color:green"><font size="5">Cell 18: Part 2 for 1176 - *function for masking body without head*</span>.

<span style="color:green"><font size="5">Cell 19: Part 3 for 1176 - *function to mask holder*</span>.

<span style="color:green"><font size="5">Cell 20: Part 4 for 1176 - *function for BMD and TMD calculation*</span>.

In [ ]:

########################################################################################################## Part 1 for 1176: function to predict the head slice ############################################
print('############## Part 1: function to predict the head slice - Running ###########')
init = time.time()

for path in df_cohort_1176['PATH']:
    predict_head_slice_1176(path,cohort_folder)
    
end = time.time()
print("############################This step took " + str(end-init) + " seconds###########################")
print('############## Part 1 - Finished ###########')
########################################################################################################## Part 2 for 1176: function for masking body without head ############################################
print('############## Part 2: function for masking body without head - Running ###########')
init = time.time()

for path in df_cohort_1176['PATH']:
    selected_row = df_cohort_1176.loc[df_cohort_1176['PATH'] == path]
    masking_predict_1176(path, int(selected_row['IMAGE_WIDTH']), int(selected_row['IMAGE_HEIGHT']),loaded_model_masking_1176)
    
end = time.time()
print("######"
      "######################This step took " + str(end-init) + " seconds###########################")
print('############## Part 2 - Finished ###########')
########################################################################################################## Part 3 for 1176: function to mask holder ############################################
print('############## Part 3: function to mask holder - Running ###########')
init = time.time()

for path in df_cohort_1176['PATH']:
    selected_row = df_cohort_1176.loc[df_cohort_1176['PATH'] == path]
    holder_masking_predict(path, int(selected_row['IMAGE_WIDTH']), int(selected_row['IMAGE_HEIGHT']),loaded_model_holder_masking_1176) 

end = time.time()
print("############################This step took " + str(end-init) + " seconds###########################")
print('############## Part 3 - Finished ###########')
########################################################################################################## Part 4 for 1176: function for BMD and TMD calculation ############################################
print('############## Part 4: function for BMD and TMD calculation - Running ###########')
init = time.time()

for path in df_cohort_1176['PATH']:
    isBMD_TMD = os.path.exists(cohort_folder+'/'+'BMD_TMD')
    if not isBMD_TMD:
        os.mkdir(cohort_folder+'/'+'BMD_TMD')
    TMD_BMD_1176(path,cohort_folder)

end = time.time()
print("############################This step took " + str(end-init) + " seconds###########################")
print('############## Part 4 - Running ###########')